# 철강 표면 결함 분류 — 최적화 실험

Baseline에서 **한 번에 한 요소씩** 변경하며 누적 비교한다.
모든 실험은 Validation으로 평가하고, 최종 선택 모델만 별도 노트북에서 Test로 평가한다.

## 실험 목록 (누적)

| 실험 | 기준 | 변경 요소 | 가설 |
|---|---|---|---|
| Baseline | - | - | 기준 성능 |
| E1 | Baseline | 입력 정규화(Normalize) | 학습 안정화, 수렴 개선 |
| E2 | E1 | 아키텍처 재설계 (Conv 3블록 · 채널 16-32-64 · 3×3 커널 · GAP) | 얕은 추출기·거대 FC 해결 → 유사 클래스 구분력 향상 |
| E3 | E2 | BatchNorm 추가 | 깊은 망 학습 안정화 + 약한 규제 |
| E4 | E3 | 데이터 증강 (flip 2종 + 90° 회전) | 1,440장 과적합 억제 (질감은 방향 불변) |
| E5 | E4 | Dropout(0.4) + Weight decay (AdamW, 1e-4) | 잔여 과적합 억제 |
| E6 | E5 | LR Scheduler (CosineAnnealing) | 후반 미세조정으로 Val 최고점 확보 |

## 규칙
- 공정 비교: 매 실험 `set_seed(SEED)` 후 학습, 동일 평가 코드
- 주 지표: **Validation Macro F1** (보조: Accuracy, 혼동행렬)
- best epoch 선택 기준: Val Macro F1
- 성능이 떨어진 변경도 표에 남기고 해석에 반영
- 각 실험 가중치는 `weights/<tag>.pt` 로 저장됨

## 0. 환경 설정

In [ ]:
import copy
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

## 1. 데이터 로드 및 분할

Baseline과 동일한 방식. `validation/images` 클래스별 60장을 30/30으로 나눠 Val·Test 구성.

In [ ]:
path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")

data_root = Path(path) / "NEU-DET"
train_root = data_root / "train" / "images"
heldout_root = data_root / "validation" / "images"

class_names = sorted(folder.name for folder in train_root.iterdir() if folder.is_dir())
rng = np.random.default_rng(SEED)

train_samples, val_samples, test_samples = [], [], []
for class_id, class_name in enumerate(class_names):
    train_files = sorted((train_root / class_name).glob("*.jpg"))
    heldout_files = sorted((heldout_root / class_name).glob("*.jpg"))
    order = rng.permutation(len(heldout_files))

    val_samples.extend((heldout_files[i], class_id) for i in order[:30])
    test_samples.extend((heldout_files[i], class_id) for i in order[30:])
    train_samples.extend((file, class_id) for file in train_files)

print("classes:", class_names)
print("split  :", len(train_samples), len(val_samples), len(test_samples))

In [ ]:
class SteelDefectDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file, label = self.samples[index]
        image = Image.open(file).convert("L")
        return self.transform(image), label

## 2. 공통 헬퍼

실험마다 바뀌는 부분(정규화·증강·모델·옵티마이저·스케줄러)만 인자로 뺐다.
`train_eval` 하나가 실험당 다음 6가지를 모두 출력한다.

1. Epoch 로그 (train/val loss·acc, val f1)
2. 학습 곡선 (Loss / Accuracy·F1)
3. Validation 성능 (Accuracy, Macro F1)
4. 혼동행렬
5. Validation 예측 결과 (클래스별 2장, 정답=검정 / 오답=빨강)
6. Feature map (첫 활성화 층 출력)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
RESULTS = []


def rot90k(img):
    """90도 배수 회전 — 강판 질감은 방향 불변이라 라벨을 깨지 않는다."""
    return transforms.functional.rotate(img, 90 * random.randint(0, 3))


def build_transforms(img_size=96, normalize=None, augment=False):
    ops = [transforms.Resize((img_size, img_size))]
    if augment:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.Lambda(rot90k),
        ]
    ops += [transforms.ToTensor()]
    if normalize is not None:
        ops += [transforms.Normalize(*normalize)]
    return transforms.Compose(ops)


def build_loaders(train_tf, eval_tf, batch_size=64):
    train_loader = DataLoader(
        SteelDefectDataset(train_samples, train_tf),
        batch_size=batch_size, shuffle=True, drop_last=True,
    )
    val_loader = DataLoader(SteelDefectDataset(val_samples, eval_tf), batch_size=batch_size)
    test_loader = DataLoader(SteelDefectDataset(test_samples, eval_tf), batch_size=batch_size)
    return train_loader, val_loader, test_loader

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = correct = count = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            count += len(images)

    return total_loss / count, correct / count


def predict(model, loader):
    model.eval()
    answers, preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            preds.extend(outputs.argmax(1).cpu().numpy())
            answers.extend(labels.numpy())
    return np.asarray(answers), np.asarray(preds)


def plot_history(history, tag):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train")
    plt.plot(epochs, history["val_loss"], marker="o", label="val")
    plt.title(f"{tag} - Loss")
    plt.xlabel("epoch")
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val acc")
    plt.plot(epochs, history["val_f1"], marker="o", label="val f1")
    plt.title(f"{tag} - Accuracy / F1")
    plt.xlabel("epoch")
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()


def show_confusion(answers, preds, title):
    ConfusionMatrixDisplay.from_predictions(
        answers, preds, display_labels=class_names, cmap="Blues", xticks_rotation=35,
    )
    plt.title(title)
    plt.tight_layout()
    plt.show()


def show_predictions(model, dataset, n_per_class=2, title="Validation predictions"):
    """클래스별 n_per_class 장을 예측. 오답은 빨간 제목."""
    model.eval()
    picks = []
    for class_id in range(len(class_names)):
        picks += [i for i, (_, label) in enumerate(dataset.samples) if label == class_id][:n_per_class]

    cols = n_per_class * 2
    rows = (len(picks) + cols - 1) // cols
    plt.figure(figsize=(3 * cols, 3 * rows))
    for position, data_index in enumerate(picks, start=1):
        image, answer = dataset[data_index]
        with torch.no_grad():
            pred = model(image.unsqueeze(0).to(device)).argmax(1).item()
        plt.subplot(rows, cols, position)
        plt.imshow(image.squeeze(), cmap="gray")
        color = "black" if pred == answer else "red"
        plt.title(f"T: {class_names[answer]}\nP: {class_names[pred]}", color=color, fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def _first_activation_layer(model):
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.LeakyReLU, nn.GELU)):
            return module
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            return module
    return None


def show_feature_maps(model, image, max_maps=8, title="Feature maps"):
    """첫 활성화 층의 출력 특징맵을 시각화."""
    layer = _first_activation_layer(model)
    store = {}
    handle = layer.register_forward_hook(
        lambda module, inputs, output: store.__setitem__("maps", output.detach().cpu())
    )
    model.eval()
    with torch.no_grad():
        model(image.unsqueeze(0).to(device))
    handle.remove()

    maps = store["maps"][0]
    count = min(max_maps, len(maps))
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 5, 1)
    plt.imshow(image.squeeze(), cmap="gray")
    plt.title("Input")
    plt.axis("off")
    for i in range(count):
        plt.subplot(2, 5, i + 2)
        plt.imshow(maps[i], cmap="viridis")
        plt.title(f"Map {i + 1}")
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def train_eval(model, train_loader, val_loader, *, tag, epochs=15, lr=1e-3,
               optimizer_name="adam", weight_decay=0.0, use_scheduler=False):
    """한 실험을 학습하고 Validation 성능을 RESULTS에 기록한다."""
    set_seed(SEED)
    model = model.to(device)

    opt_cls = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW}[optimizer_name]
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        if use_scheduler else None
    )

    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"]}
    best_f1, best_state = -1.0, None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        val_answers, val_preds = predict(model, val_loader)
        val_f1 = f1_score(val_answers, val_preds, average="macro")
        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:02d} | train {train_loss:.3f}/{train_acc:.3f} | "
            f"val {val_loss:.3f}/{val_acc:.3f} | val_f1 {val_f1:.3f}"
        )

    model.load_state_dict(best_state)
    val_answers, val_preds = predict(model, val_loader)
    val_accuracy = accuracy_score(val_answers, val_preds)
    val_macro_f1 = f1_score(val_answers, val_preds, average="macro")

    Path("weights").mkdir(exist_ok=True)
    torch.save(best_state, f"weights/{tag}.pt")
    RESULTS.append({"tag": tag, "val_acc": round(val_accuracy, 4), "val_f1": round(val_macro_f1, 4)})

    print(f"\n[{tag}] Val Accuracy = {val_accuracy:.4f} | Val Macro F1 = {val_macro_f1:.4f}")
    plot_history(history, tag)
    show_confusion(val_answers, val_preds, f"{tag} - Validation Confusion Matrix")
    show_predictions(model, val_loader.dataset, title=f"{tag} - Validation predictions")
    sample_image, _ = val_loader.dataset[0]
    show_feature_maps(model, sample_image, title=f"{tag} - Feature maps (first activation)")
    return model, history

## 3. 입력 정규화 통계

train 셋에서만 채널 평균·표준편차를 계산한다 (데이터 누수 방지).

In [ ]:
stats_loader = DataLoader(SteelDefectDataset(train_samples, build_transforms()), batch_size=64)

pixel_sum = pixel_sq_sum = pixel_count = 0.0
for images, _ in stats_loader:
    pixel_sum += images.sum().item()
    pixel_sq_sum += (images ** 2).sum().item()
    pixel_count += images.numel()

NORM_MEAN = pixel_sum / pixel_count
NORM_STD = (pixel_sq_sum / pixel_count - NORM_MEAN ** 2) ** 0.5
NORM = ((NORM_MEAN,), (NORM_STD,))
print(f"train mean = {NORM_MEAN:.4f}, std = {NORM_STD:.4f}")

## 4. 모델 정의

In [ ]:
class BaselineCNN(nn.Module):
    """제공된 Baseline 그대로: Conv 1블록 + 큰 FC."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8 * 46 * 46, 32),
            nn.ReLU(),
            nn.Linear(32, 6),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class SteelCNN(nn.Module):
    """설정 가능한 CNN: 블록 수(채널 리스트), BatchNorm, Dropout, Activation."""

    def __init__(self, channels=(16, 32, 64), use_bn=False, dropout=0.0,
                 activation="relu", num_classes=6, in_channels=1):
        super().__init__()
        act = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU, "gelu": nn.GELU}[activation]

        blocks = []
        prev = in_channels
        for channel in channels:
            blocks.append(nn.Conv2d(prev, channel, kernel_size=3, padding=1))
            if use_bn:
                blocks.append(nn.BatchNorm2d(channel))
            blocks.append(act())
            blocks.append(nn.MaxPool2d(kernel_size=2))
            prev = channel

        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(prev, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

## 5. Baseline

변경 없음. 기준 성능과 학습 곡선 확인.

In [ ]:
plain_tf = build_transforms()
train_loader, val_loader, test_loader = build_loaders(plain_tf, plain_tf, batch_size=64)

baseline_model, _ = train_eval(BaselineCNN(), train_loader, val_loader, tag="baseline", epochs=15)

## 6. E1 — 입력 정규화

- 변경: `transforms.Normalize(mean, std)` 추가 (train/val 동일)
- 가설: 입력 분포를 표준화하면 학습 초반이 안정되고 수렴이 빨라진다
- 확인: 학습 곡선 초반 기울기, Val Macro F1

In [ ]:
norm_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(norm_tf, norm_tf, batch_size=64)

e1_model, _ = train_eval(BaselineCNN(), train_loader, val_loader, tag="E1_normalize", epochs=15)

## 7. E2 — 아키텍처 재설계

- 변경: Conv 1블록 → **3블록 (16-32-64) + 3×3 커널 + GAP**, 큰 FC 제거
- 가설: 계층적 특징 추출로 유사 클래스(`crazing`↔`rolled-in_scale` 등) 구분력이 오른다.
  GAP로 파라미터가 급감해 과적합도 완화된다
- 확인: Val Macro F1 상승폭, 혼동행렬의 유사 클래스 칸, 파라미터 수

In [ ]:
model = SteelCNN(channels=(16, 32, 64), use_bn=False)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

norm_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(norm_tf, norm_tf, batch_size=64)

e2_model, _ = train_eval(model, train_loader, val_loader, tag="E2_arch", epochs=15)

## 8. E3 — BatchNorm

- 변경: 각 Conv 뒤에 `BatchNorm2d` 추가 (기준: E2)
- 가설: 층별 분포가 안정되어 학습이 빨라지고, 미니배치 통계 노이즈가 약한 규제로 작용한다
- 확인: 수렴 속도(곡선), train-val 격차, Val Macro F1

In [ ]:
norm_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(norm_tf, norm_tf, batch_size=64)

e3_model, _ = train_eval(
    SteelCNN(channels=(16, 32, 64), use_bn=True),
    train_loader, val_loader, tag="E3_batchnorm", epochs=15,
)

## 9. E4 — 데이터 증강

- 변경: **train 로더에만** 좌우/상하 flip + 90° 배수 회전 (기준: E3, val/test는 그대로)
- 가설: 강판 질감은 방향 불변이라 flip/회전이 라벨을 깨지 않고 데이터 다양성을 크게 늘린다 → 과적합 감소
- 확인: train-val 격차 축소, Val Macro F1
- 증강으로 수렴이 느려지므로 epochs를 늘린다

In [ ]:
aug_tf = build_transforms(normalize=NORM, augment=True)
eval_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(aug_tf, eval_tf, batch_size=64)

e4_model, _ = train_eval(
    SteelCNN(channels=(16, 32, 64), use_bn=True),
    train_loader, val_loader, tag="E4_augment", epochs=20,
)

## 10. E5 — Dropout + Weight decay

- 변경: 분류기 앞 `Dropout(0.4)` + 옵티마이저 `AdamW(weight_decay=1e-4)` (기준: E4)
- 가설: 증강 후 남은 과적합을 파라미터 규제로 추가 억제한다
- 확인: train-val 격차, Val Macro F1 (과하면 과소적합으로 오히려 하락)

In [ ]:
aug_tf = build_transforms(normalize=NORM, augment=True)
eval_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(aug_tf, eval_tf, batch_size=64)

e5_model, _ = train_eval(
    SteelCNN(channels=(16, 32, 64), use_bn=True, dropout=0.4),
    train_loader, val_loader, tag="E5_dropout_wd",
    epochs=20, optimizer_name="adamw", weight_decay=1e-4,
)

## 11. E6 — LR Scheduler

- 변경: `CosineAnnealingLR` 추가 + epochs 30 (기준: E5)
- 가설: 후반에 학습률을 낮춰 미세조정하면 Val 최고점이 올라가고 곡선 등락이 준다
- 확인: 후반 곡선 안정성, Val Macro F1 최고값

In [ ]:
aug_tf = build_transforms(normalize=NORM, augment=True)
eval_tf = build_transforms(normalize=NORM)
train_loader, val_loader, test_loader = build_loaders(aug_tf, eval_tf, batch_size=64)

e6_model, _ = train_eval(
    SteelCNN(channels=(16, 32, 64), use_bn=True, dropout=0.4),
    train_loader, val_loader, tag="E6_scheduler",
    epochs=30, optimizer_name="adamw", weight_decay=1e-4, use_scheduler=True,
)

## 12. 결과 비교

In [ ]:
results_df = pd.DataFrame(RESULTS).set_index("tag")
results_df["delta_f1"] = results_df["val_f1"].diff().fillna(0).round(4)
results_df

In [ ]:
ax = results_df["val_f1"].plot(kind="bar", figsize=(9, 4), color="steelblue")
ax.set_ylabel("Validation Macro F1")
ax.set_ylim(0, 1)
ax.set_title("experiment vs Validation Macro F1")
for i, value in enumerate(results_df["val_f1"]):
    ax.text(i, value + 0.01, f"{value:.3f}", ha="center")
plt.tight_layout()
plt.show()

## 13. 결과 해석 (작성)

| 실험 | 변경 | Val Acc | Val F1 | ΔF1 | 해석 |
|---|---|---:|---:|---:|---|
| Baseline | - |  |  | - |  |
| E1 | Normalize |  |  |  |  |
| E2 | Conv 3블록 + GAP |  |  |  |  |
| E3 | BatchNorm |  |  |  |  |
| E4 | 데이터 증강 |  |  |  |  |
| E5 | Dropout + Weight decay |  |  |  |  |
| E6 | LR Scheduler |  |  |  |  |

- 성능이 하락한 실험과 되돌릴지 여부(근거):
- 가장 효과가 큰 변경:
- 최종 선택 모델: `weights/<tag>.pt` — 선정 근거:

## 다음 단계
`weights/` 에서 Val Macro F1이 가장 높은 가중치를 골라 **최종 모델 노트북**
(`조원1_조원2_최종모델.ipynb`)에서 Test로 1회만 평가한다.